In [1]:
"""
ApexPlanet Data Analytics Internship - Task 2
PART 2: SQL for Business Questions
Loads the cleaned dataset into an in-memory SQLite DB and answers
7 business questions using SQL (filtering, aggregation, joins).
"""

import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv('cleaned_dataset.csv')

In [3]:
# ─────────────────────────────────────────────
# SETUP: Build a small relational schema (3 tables) so we can
# demonstrate JOINs, not just a single flat table.
# ─────────────────────────────────────────────
conn = sqlite3.connect(':memory:')

# Orders fact table
orders = df[['Order_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category',
             'Quantity', 'Unit_Price', 'Total_Sales', 'Order_Date_Year',
             'Order_Date_Month', 'Purchase_Month_Name','Revenue_Category']]
orders.to_sql('orders', conn, index=False, if_exists='replace')

# Customers dimension table (deduplicated)
customers = df[['Customer_ID', 'Customer_Name', 'Age', 'Age_Group', 'Gender', 'City']]\
              .drop_duplicates(subset='Customer_ID')
customers.to_sql('customers', conn, index=False, if_exists='replace')

# Products dimension table (deduplicated)
products = df[['Product', 'Category']].drop_duplicates(subset='Product')
products.to_sql('products', conn, index=False, if_exists='replace')

cur = conn.cursor()
def run_query(title, sql):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(sql.strip())
    print("-" * 70)
    result = pd.read_sql_query(sql, conn)
    print(result.to_string(index=False))
    return result

results = {}


In [4]:
# ─────────────────────────────────────────────
# Q1: Top 5 products by revenue
# ─────────────────────────────────────────────
sql1 = """
SELECT
    Product,
    Category,
    SUM(Total_Sales)          AS Total_Revenue,
    SUM(Quantity)             AS Units_Sold,
    COUNT(*)                  AS Order_Count
FROM orders
GROUP BY Product, Category
ORDER BY Total_Revenue DESC
LIMIT 5;
"""
results['Q1_top5_products'] = run_query("Q1: What are the top 5 products by revenue?", sql1)


Q1: What are the top 5 products by revenue?
SELECT
    Product,
    Category,
    SUM(Total_Sales)          AS Total_Revenue,
    SUM(Quantity)             AS Units_Sold,
    COUNT(*)                  AS Order_Count
FROM orders
GROUP BY Product, Category
ORDER BY Total_Revenue DESC
LIMIT 5;
----------------------------------------------------------------------
Product    Category  Total_Revenue  Units_Sold  Order_Count
 Laptop Electronics    25443008.51         970          170
 Mobile Electronics    25335573.19        1008          184
   Book   Education    25031689.40         977          178
   Rice     Grocery    22231711.28         826          153
  Chair   Furniture    21521561.48         855          159


In [5]:
sql2 = """
SELECT
    Order_Date_Year,
    Order_Date_Month,
    Purchase_Month_Name,
    SUM(Total_Sales) AS Monthly_Revenue,
    COUNT(*) AS Order_Count
FROM orders
GROUP BY
    Order_Date_Year,
    Order_Date_Month,
    Purchase_Month_Name
ORDER BY
    Order_Date_Year,
    Order_Date_Month;
"""
results['Q2_monthly_trend'] = run_query(
    "Q2: What is the monthly sales trend?",
    sql2
)


Q2: What is the monthly sales trend?
SELECT
    Order_Date_Year,
    Order_Date_Month,
    Purchase_Month_Name,
    SUM(Total_Sales) AS Monthly_Revenue,
    COUNT(*) AS Order_Count
FROM orders
GROUP BY
    Order_Date_Year,
    Order_Date_Month,
    Purchase_Month_Name
ORDER BY
    Order_Date_Year,
    Order_Date_Month;
----------------------------------------------------------------------
 Order_Date_Year  Order_Date_Month Purchase_Month_Name  Monthly_Revenue  Order_Count
            2025                 1             January      10096190.73           76
            2025                 2            February      11511181.46           86
            2025                 3               March      13059899.94           89
            2025                 4               April      12222700.17           80
            2025                 5                 May      10984689.25           85
            2025                 6                June      12912332.64           83
            

In [6]:
# ─────────────────────────────────────────────
# Q3: Which city generates the highest revenue?
# ─────────────────────────────────────────────
sql3 = """
SELECT
    c.City,
    SUM(o.Total_Sales) AS Total_Revenue,
    COUNT(DISTINCT o.Customer_ID) AS Unique_Customers,
    COUNT(*) AS Order_Count,
    ROUND(AVG(o.Total_Sales), 2) AS Avg_Order_Value
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.City
ORDER BY Total_Revenue DESC;
"""
results['Q3_revenue_by_city'] = run_query(
    "Q3: Which city generates the highest revenue?",
    sql3
)



Q3: Which city generates the highest revenue?
SELECT
    c.City,
    SUM(o.Total_Sales) AS Total_Revenue,
    COUNT(DISTINCT o.Customer_ID) AS Unique_Customers,
    COUNT(*) AS Order_Count,
    ROUND(AVG(o.Total_Sales), 2) AS Avg_Order_Value
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.City
ORDER BY Total_Revenue DESC;
----------------------------------------------------------------------
     City  Total_Revenue  Unique_Customers  Order_Count  Avg_Order_Value
    Patna    20845665.07               139          148        140849.09
  Kolkata    20125156.04               129          138        145834.46
Bengaluru    18280248.17               116          123        148619.90
   Mumbai    17392434.13               123          128        135878.39
    Delhi    17272358.12               117          127        136002.82
Hyderabad    16806215.82               119          124        135534.00
     Pune    14510083.00                92           96        151

In [7]:
# ─────────────────────────────────────────────
# Q4: Which age group spends the most on average?
# ─────────────────────────────────────────────
sql4 = """
SELECT
    c.Age_Group,
    COUNT(*) AS Order_Count,
    SUM(o.Total_Sales) AS Total_Revenue,
    ROUND(AVG(o.Total_Sales), 2) AS Avg_Order_Value
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Age_Group
ORDER BY Avg_Order_Value DESC;
"""
results['Q4_age_group_spend'] = run_query(
    "Q4: Which age group has the highest average order value?",
    sql4
)



Q4: Which age group has the highest average order value?
SELECT
    c.Age_Group,
    COUNT(*) AS Order_Count,
    SUM(o.Total_Sales) AS Total_Revenue,
    ROUND(AVG(o.Total_Sales), 2) AS Avg_Order_Value
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Age_Group
ORDER BY Avg_Order_Value DESC;
----------------------------------------------------------------------
  Age_Group  Order_Count  Total_Revenue  Avg_Order_Value
      Youth           17     3011623.72        177154.34
     Senior          198    28055564.40        141694.77
      Adult          422    58764850.11        139253.20
Young Adult          363    49567401.42        136549.32


In [8]:
# ─────────────────────────────────────────────
# Q5: Category performance split by gender
# ─────────────────────────────────────────────
sql5 = """
SELECT
    o.Category,
    c.Gender,
    SUM(o.Total_Sales) AS Total_Revenue,
    COUNT(*) AS Order_Count
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY o.Category, c.Gender
ORDER BY o.Category, Total_Revenue DESC;
"""
results['Q5_category_by_gender'] = run_query(
    "Q5: How does category performance differ by gender?",
    sql5)



Q5: How does category performance differ by gender?
SELECT
    o.Category,
    c.Gender,
    SUM(o.Total_Sales) AS Total_Revenue,
    COUNT(*) AS Order_Count
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY o.Category, c.Gender
ORDER BY o.Category, Total_Revenue DESC;
----------------------------------------------------------------------
   Category Gender  Total_Revenue  Order_Count
  Education Female    13023326.51           96
  Education   Male    12008362.89           82
Electronics   Male    27973691.21          196
Electronics Female    22804890.49          158
    Fashion Female    11228048.87           88
    Fashion   Male     8607846.92           68
  Furniture   Male    11079708.15           80
  Furniture Female    10441853.33           79
    Grocery   Male    12797731.03           83
    Grocery Female     9433980.25           70


In [9]:
# ─────────────────────────────────────────────
# Q6: Top 5 highest-value customers
# ─────────────────────────────────────────────
sql6 = """
SELECT
    c.Customer_ID,
    c.Customer_Name,
    c.City,
    COUNT(o.Order_ID) AS Total_Orders,
    SUM(o.Total_Sales) AS Lifetime_Value
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Customer_ID, c.Customer_Name, c.City
ORDER BY Lifetime_Value DESC
LIMIT 5;
"""
results['Q6_top5_customers'] = run_query(
    "Q6: Who are the top 5 highest-value customers?",
    sql6)



Q6: Who are the top 5 highest-value customers?
SELECT
    c.Customer_ID,
    c.Customer_Name,
    c.City,
    COUNT(o.Order_ID) AS Total_Orders,
    SUM(o.Total_Sales) AS Lifetime_Value
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Customer_ID, c.Customer_Name, c.City
ORDER BY Lifetime_Value DESC
LIMIT 5;
----------------------------------------------------------------------
Customer_ID Customer_Name      City  Total_Orders  Lifetime_Value
   Cust9510  Customer_345   Kolkata             2       867333.24
   Cust6845  Customer_337 Hyderabad             2       769479.75
   Cust6532  Customer_348      Pune             2       610981.74
   Cust6082  Customer_301    Mumbai             2       548416.39
   Cust3689  Customer_179     Delhi             2       542484.08


In [10]:
# ─────────────────────────────────────────────
# Q7: Premium-tier orders in Electronics category (multi-condition filter + join)
# ─────────────────────────────────────────────
sql7 = """
SELECT
    o.Order_ID,
    o.Product,
    c.Customer_Name,
    c.City,
    o.Quantity,
    o.Unit_Price,
    o.Total_Sales,
    o.Order_Date
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Category = 'Electronics'
  AND o.Revenue_Category = 'High_Revenue'
ORDER BY o.Total_Sales DESC;
"""
results['Q7_premium_electronics'] = run_query(
    "Q7: Which orders are High-Revenue purchases in the Electronics category?", sql7)



Q7: Which orders are High-Revenue purchases in the Electronics category?
SELECT
    o.Order_ID,
    o.Product,
    c.Customer_Name,
    c.City,
    o.Quantity,
    o.Unit_Price,
    o.Total_Sales,
    o.Order_Date
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Category = 'Electronics'
  AND o.Revenue_Category = 'High_Revenue'
ORDER BY o.Total_Sales DESC;
----------------------------------------------------------------------
 Order_ID Product Customer_Name      City  Quantity  Unit_Price  Total_Sales Order_Date
Ord100764  Laptop   Customer_34     Delhi        10    49217.41    492174.10 2025-10-13
Ord100974  Laptop  Customer_392   Kolkata        10    48566.78    485667.80 2025-07-28
Ord100366  Mobile  Customer_230 Bengaluru        10    47493.59    474935.90 2025-03-13
Ord100714  Mobile  Customer_205   Kolkata        10    47079.86    470798.60 2025-05-05
Ord100723  Laptop  Customer_495     Delhi        10    44401.64    444016.40 2025-11-28
Ord100434  Mobile 